# Inpainting submarino -> tokens DINOv3-VQ (Kaggle)
Requisitos: GPU (T4/P100), Internet ON, secret `HF_TOKEN` (acceso aprobado a `facebook/dinov3-vitb16-pretrain-lvd1689m`),
y un Dataset con: `common.py train.py predict.py dino_vq.py codebook.npy target.csv train.zip test.zip` (o carpetas `train/` `test/`).
Ejecutar con **Save & Run All** (commit en segundo plano).

In [ ]:
import os, glob, subprocess, shutil, sys
# --- Config ---
SMOKE = False            # True: prueba rápida (60 pasos) antes del run largo
TIME_BUDGET_H = 3      # el commit de Kaggle dura máx ~12h; deja margen para predict
RESUME_FROM = ""         # p.ej. /kaggle/input/<ckpt-dataset>/last.pt para continuar un entrenamiento
# --- Localizar dataset ---
hits = glob.glob('/kaggle/input/datasets/pieropr2000/aquatic/target.csv', recursive=True)
#hits = glob.glob('/kaggle/input/**/target.csv', recursive=True)
assert hits, 'No encuentro target.csv en /kaggle/input'
SRC = os.path.dirname(hits[0]); print('SRC =', SRC)
W = '/kaggle/working'; T = '/kaggle/temp'; os.makedirs(T, exist_ok=True)
for f in ['common.py','train.py','predict.py','dino_vq.py','diagnose.py']:
    shutil.copy(glob.glob(f'/kaggle/input/**/{f}', recursive=True)[0], W)
shutil.copy(f'{SRC}/codebook.npy', W); shutil.copy(f'{SRC}/target.csv', W)
os.chdir(W)

SRC = /kaggle/input/datasets/pieropr2000/aquatic


In [2]:
def find_dir(name):
    d = f'{SRC}/{name}'
    if os.path.isdir(d): return d
    z = f'{SRC}/{name}.zip'
    if os.path.exists(z):
        subprocess.run(['unzip','-q','-o',z,'-d',f'{T}/{name}_x'], check=True)
        sub = f'{T}/{name}_x/{name}'
        return sub if os.path.isdir(sub) else f'{T}/{name}_x'
    raise FileNotFoundError(name)
TRAIN, TEST = find_dir('train'), find_dir('test')
print(TRAIN, len(os.listdir(TRAIN)), '|', TEST, len(os.listdir(TEST)))

/kaggle/input/datasets/pieropr2000/aquatic/train 1 | /kaggle/input/datasets/pieropr2000/aquatic/test 1


In [3]:
!pip install -q -U transformers
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = secret_value_0
os.environ["HUGGING_FACE_HUB_TOKEN"] = secret_value_0
login(secret_value_0)

import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), "GPU(s):", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2.11.0+cu130 True Tesla T4


In [4]:
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["NCCL_P2P_DISABLE"] = "1"   # descomentar si el run con 2 GPUs se cuelga al iniciar

BS = 12                  # POR GPU (con 2 T4: batch efectivo 24). Si hay OOM, bajar a 8
ngpu = torch.cuda.device_count()
launcher = f"torchrun --standalone --nproc_per_node {ngpu}" if ngpu > 1 else "python -u"

cmd = f"{launcher} train.py --train-dir {TRAIN} --codebook codebook.npy --out runs/v1 --cache {T}/train.npy --bs {BS} --mask-csv target.csv --p-real 0.5 --group-split 1 --photo-aug 0.8"
if SMOKE: cmd += " --max-steps 60 --val-n 50 --val-every 30 --time-budget-h 0.2"
else:     cmd += f" --time-budget-h {TIME_BUDGET_H}"
if RESUME_FROM: cmd += f" --resume {RESUME_FROM}"
print(cmd)
!{cmd}

In [ ]:
# Diagnóstico opcional: máscaras reales vs uniformes y fuga train/val (usa runs/v1/last.pt)
!python diagnose.py --cache {T}/train.npy --test {TEST} --ckpt runs/v1/last.pt

In [5]:
!python predict.py --ckpt runs/v1/last.pt --test {TEST} --target target.csv --out generate --tta

  0%|                                                  | 0/1056 [00:00<?, ?it/s]/kaggle/working/predict.py:31: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(orig).to(dev).permute(2, 0, 1)[None].float() / 255
100%|███████████████████████████████████████| 1056/1056 [02:34<00:00,  6.83it/s]
listo -> generate


In [6]:
!python dino_vq.py --images-root generate --target-csv target.csv --codebook-path codebook.npy --output-csv submission.csv
import pandas as pd
s = pd.read_csv('submission.csv'); print(s.shape, s.Target.min(), s.Target.max(), s.Id.duplicated().sum())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Generando submission: 100%|███████████████████| 132/132 [00:07<00:00, 16.66it/s]
Archivo generado: submission.csv
Filas: 14913
(14913, 2) 0 1023 0


In [7]:
# Opcional: liberar espacio de salida (solo quedan last.pt y submission.csv)
shutil.rmtree('generate', ignore_errors=True)